# Chronos-2 Electricity Forecasting: Final Test Evaluation

## Overview

This notebook performs the final evaluation of the LoRA fine-tuned Chronos-2 model on the unseen test dataset.

The main tasks performed in this notebook are:

- Setting up the environment and loading the prepared dataset splits.
- Loading the fine-tuned LoRA adapter weights.
- Generating 24-hour rolling-origin forecasts for all 321 customer series in the test set.
- Calculating final MAE, RMSE, and MASE evaluation metrics.
- Comparing the fine-tuned model against the zero-shot and seasonal naive baselines.
- Saving the final test predictions, comparative metrics, and computational costs.

In [ ]:
!pip install -q chronos-forecasting peft
!pip install --upgrade torchao==0.18.0 --index-url https://download.pytorch.org/whl/cu130

## 1. Environment Setup and Data Loading

Google Drive is mounted to access the dataset splits saved during the data preparation stage. The training, validation, and test sets are loaded to prepare the historical context and target values for the final evaluation.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch
import json
import time
import os

print("GPU available:", torch.cuda.is_available())

Mounted at /content/drive
GPU available: True


In [2]:
data_dir = "/content/drive/MyDrive/Chronos2_Finetune"

saved = np.load(f"{data_dir}/dataset_splits.npz")

train_data = saved["train"]
val_data = saved["val"]
test_data = saved["test"]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (321, 18412)
Validation: (321, 3946)
Test: (321, 3946)


## 2. Loading the Fine-Tuned Model and Baselines

The fine-tuned LoRA adapter is loaded onto the available CUDA device.

Additionally, the baseline evaluation metrics (zero-shot and seasonal naive) calculated in the previous evaluation stage are uploaded and loaded into a DataFrame for the final comparison.

In [7]:
from chronos import Chronos2Pipeline

checkpoint_path = (
    "/content/drive/MyDrive/Chronos2_Finetune/"
    "Chronos2_LoRA_Checkpoint/finetuned-ckpt"
)

lora_pipeline = Chronos2Pipeline.from_pretrained(
    checkpoint_path,
    device_map="cuda"
)

print("LoRA checkpoint loaded successfully")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

W1111 07:15:59.107000 2547 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


LoRA checkpoint loaded successfully


In [8]:
print("Model:", type(lora_pipeline).__name__)
print("Device:", next(lora_pipeline.model.parameters()).device)
print("Checkpoint:", checkpoint_path)

Model: Chronos2Pipeline
Device: cuda:0
Checkpoint: /content/drive/MyDrive/Chronos2_Finetune/Chronos2_LoRA_Checkpoint/finetuned-ckpt


In [9]:
uploaded = files.upload()

Saving baseline_comparison.csv to baseline_comparison.csv


In [10]:
baseline = pd.read_csv("baseline_comparison.csv")

print("Shape:", baseline.shape)
print("\nColumns:", baseline.columns.tolist())
print("\nFirst five rows:")
display(baseline.head())

Shape: (321, 7)

Columns: ['series_id', 'MAE_chronos', 'RMSE_chronos', 'MASE', 'MAE_naive', 'RMSE_naive', 'Chronos_better']

First five rows:


,series_id,MAE_chronos,RMSE_chronos,MASE,MAE_naive,RMSE_naive,Chronos_better
0,T1,5.168531,12.620989,0.658799,5.822155,13.872331,True
1,T2,4.162787,7.317110,0.419868,5.912856,11.858573,True
2,T3,0.580795,1.433557,0.059256,0.901677,2.070468,True
3,T4,37.449329,64.105268,0.784030,48.259399,72.512354,True
4,T5,22.116941,32.793123,0.873743,25.761179,37.279045,True


## 3. Test Data Preparation and Inference

The seasonal scales are calculated directly from the training data to be used later for the MASE metric.

To evaluate the model, rolling 24-hour forecast windows are generated across the entire test period. For each window, the most recent 168 hours of data (combining training and previously observed test data) are provided as historical context.

Inference is then performed in batches to generate the median (0.5 quantile) 24-hour forecasts for all test windows.

In [11]:
seasonal_scales = np.array([
    np.mean(np.abs(train_data[i][24:] - train_data[i][:-24]))
    for i in range(len(train_data))
])

print("Number of seasonal scales:", len(seasonal_scales))
print("Baseline rows:", len(baseline))
print("Test observations per series:", test_data.shape[1])
print("Complete 24-hour forecast windows:", test_data.shape[1] // 24)
print("Total forecast windows:", len(test_data) * (test_data.shape[1] // 24))
print("Undefined seasonal scales:", np.sum(seasonal_scales == 0))

Number of seasonal scales: 321
Baseline rows: 321
Test observations per series: 3946
Complete 24-hour forecast windows: 164
Total forecast windows: 52644
Undefined seasonal scales: 0


In [12]:
quantiles = lora_pipeline.model.config.chronos_config["quantiles"]
median_index = quantiles.index(0.5)

print("Available quantiles:", quantiles)
print("Median index:", median_index)

Available quantiles: [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
Median index: 10


In [13]:
# Store the historical context, actual forecast targets, and corresponding customer index for every forecast window.
contexts = []
actuals = []
series_indices = []

for i in range(len(test_data)):

    # Generate forecast origins at 24-hour intervals.
    # Each forecast covers the next 24 hours.
    for start in range(0, len(test_data[i]) - 24 + 1, 24):

        end = start + 24

        # Combine training data with previously observed test data.
        # Keep only the most recent 168 hours (7 days) as context.
        context = np.concatenate([
            train_data[i],
            test_data[i][:start]
        ])[-168:]

        actual = test_data[i][start:end]

        contexts.append(context)
        actuals.append(actual)
        series_indices.append(i)

contexts = np.array(contexts, dtype=np.float32)
actuals = np.array(actuals, dtype=np.float32)
series_indices = np.array(series_indices)

print("Contexts:", contexts.shape)
print("Actuals:", actuals.shape)
print("Series indices:", series_indices.shape)

Contexts: (52644, 168)
Actuals: (52644, 24)
Series indices: (52644,)


In [15]:
batch_size = 32
lora_predictions = []

torch.cuda.synchronize()
inference_start = time.time()

# Process the test contexts in batches.
for start in range(0, len(contexts), batch_size):

    end = min(start + batch_size, len(contexts))
    context_tensor = torch.tensor(
        contexts[start:end],
        dtype=torch.float32
    ).unsqueeze(1)

    # Generate 24-hour forecasts.
    forecasts = lora_pipeline.predict(
        context_tensor,
        prediction_length=24
    )

    # Extract the median (0.5 quantile) forecast for each series.
    for forecast in forecasts:
        median_forecast = forecast[0, median_index, :].cpu().numpy()
        lora_predictions.append(median_forecast)

    if (start // batch_size + 1) % 100 == 0:
        print(f"Processed {end}/{len(contexts)} windows")

torch.cuda.synchronize()
inference_time = time.time() - inference_start

lora_predictions = np.array(lora_predictions)

print("\nPrediction shape:", lora_predictions.shape)
print(f"Inference time: {inference_time / 60:.2f} minutes")

Processed 3200/52644 windows
Processed 6400/52644 windows
Processed 9600/52644 windows
Processed 12800/52644 windows
Processed 16000/52644 windows
Processed 19200/52644 windows
Processed 22400/52644 windows
Processed 25600/52644 windows
Processed 28800/52644 windows
Processed 32000/52644 windows
Processed 35200/52644 windows
Processed 38400/52644 windows
Processed 41600/52644 windows
Processed 44800/52644 windows
Processed 48000/52644 windows
Processed 51200/52644 windows

Prediction shape: (52644, 24)
Inference time: 1.58 minutes


## 4. Final Evaluation and Baseline Comparison

Forecast errors (MAE, RMSE, and MASE) are calculated for the fine-tuned model's predictions. These metrics are aggregated per customer series and merged with the previously uploaded baseline results.

The performance of the LoRA fine-tuned model is then directly compared against both the zero-shot Chronos-2 model and the seasonal naive baseline to quantify the overall improvement.

In [16]:
absolute_errors = np.abs(actuals - lora_predictions)
squared_errors = (actuals - lora_predictions) ** 2

window_mae = absolute_errors.mean(axis=1)
window_rmse = np.sqrt(squared_errors.mean(axis=1))

print("Number of forecast windows:", len(window_mae))
print("Mean window MAE:", window_mae.mean())
print("Mean window RMSE:", window_rmse.mean())

Number of forecast windows: 52644
Mean window MAE: 170.7693
Mean window RMSE: 228.7855


In [17]:
lora_results = []

for i in range(len(train_data)):

    # Select all forecast windows belonging to this customer.
    customer_mask = series_indices == i

    customer_actuals = actuals[customer_mask]
    customer_predictions = lora_predictions[customer_mask]

    # Calculate errors across all 164 forecast windows (3936 values).
    mae = np.mean(np.abs(customer_actuals - customer_predictions))

    rmse = np.sqrt(
        np.mean((customer_actuals - customer_predictions) ** 2)
    )

    # Calculate MASE using the training-only seasonal scale.
    mase = mae / seasonal_scales[i]

    # Store the customer's evaluation metrics.
    lora_results.append({
        "series_id": f"T{i + 1}",
        "LoRA_MAE": mae,
        "LoRA_RMSE": rmse,
        "LoRA_MASE": mase
    })

lora_df = pd.DataFrame(lora_results)

print("Shape:", lora_df.shape)
display(lora_df.head())

Shape: (321, 4)


,series_id,LoRA_MAE,LoRA_RMSE,LoRA_MASE
0,T1,5.165778,12.619461,0.658448
1,T2,4.165598,7.326566,0.420152
2,T3,0.581147,1.433270,0.059292
3,T4,37.389442,64.159691,0.782776
4,T5,22.107355,32.820972,0.873364


In [18]:
# Calculate seasonal-naive MASE using the same training-only scales.
baseline["Naive_MASE"] = (
    baseline["MAE_naive"] / seasonal_scales
)

comparison = baseline.merge(
    lora_df,
    on="series_id",
    how="inner"
)

print("Comparison shape:", comparison.shape)
print("\nMissing values:")
print(comparison.isnull().sum())

display(comparison.head())

Comparison shape: (321, 11)

Missing values:
series_id         0
MAE_chronos       0
RMSE_chronos      0
MASE              0
MAE_naive         0
RMSE_naive        0
Chronos_better    0
Naive_MASE        0
LoRA_MAE          0
LoRA_RMSE         0
LoRA_MASE         0
dtype: int64


,series_id,MAE_chronos,RMSE_chronos,MASE,MAE_naive,RMSE_naive,Chronos_better,Naive_MASE,LoRA_MAE,LoRA_RMSE,LoRA_MASE
0,T1,5.168531,12.620989,0.658799,5.822155,13.872331,True,0.742112,5.165778,12.619461,0.658448
1,T2,4.162787,7.317110,0.419868,5.912856,11.858573,True,0.596384,4.165598,7.326566,0.420152
2,T3,0.580795,1.433557,0.059256,0.901677,2.070468,True,0.091995,0.581147,1.433270,0.059292
3,T4,37.449329,64.105268,0.784030,48.259399,72.512354,True,1.010347,37.389442,64.159691,0.782776
4,T5,22.116941,32.793123,0.873743,25.761179,37.279045,True,1.017711,22.107355,32.820972,0.873364


In [19]:
models = {
    "Chronos-2 Zero-Shot": ["MAE_chronos", "RMSE_chronos", "MASE"],
    "Chronos-2 LoRA": ["LoRA_MAE", "LoRA_RMSE", "LoRA_MASE"],
    "Seasonal Naive": ["MAE_naive", "RMSE_naive", "Naive_MASE"]
}

summary = {}

for model, columns in models.items():
    summary[model] = {
        "Mean MAE": comparison[columns[0]].mean(),
        "Median MAE": comparison[columns[0]].median(),
        "Mean RMSE": comparison[columns[1]].mean(),
        "Median RMSE": comparison[columns[1]].median(),
        "Mean MASE": comparison[columns[2]].mean(),
        "Median MASE": comparison[columns[2]].median()
    }

summary_df = pd.DataFrame(summary).T

display(summary_df.round(4))

,Mean MAE,Median MAE,Mean RMSE,Median RMSE,Mean MASE,Median MASE
Chronos-2 Zero-Shot,170.4269,43.1387,323.5805,70.7897,0.8181,0.8233
Chronos-2 LoRA,170.7693,43.1383,324.0206,70.7994,0.8179,0.8230
Seasonal Naive,210.5993,54.5094,374.2500,95.8477,1.0412,1.0177


In [20]:
# Compare LoRA against zero-shot and seasonal naive.
lora_wins_zero = (
    comparison["LoRA_MAE"] < comparison["MAE_chronos"]
).sum()

zero_wins_lora = (
    comparison["MAE_chronos"] < comparison["LoRA_MAE"]
).sum()

ties_lora_zero = (
    comparison["LoRA_MAE"] == comparison["MAE_chronos"]
).sum()

lora_wins_naive = (
    comparison["LoRA_MAE"] < comparison["MAE_naive"]
).sum()

zero_wins_naive = (
    comparison["MAE_chronos"] < comparison["MAE_naive"]
).sum()

print("LoRA vs Zero-Shot")
print("LoRA wins:", lora_wins_zero)
print("Zero-Shot wins:", zero_wins_lora)
print("Ties:", ties_lora_zero)

print("\nAgainst Seasonal Naive")
print("LoRA wins:", lora_wins_naive)
print("Zero-Shot wins:", zero_wins_naive)

LoRA vs Zero-Shot
LoRA wins: 208
Zero-Shot wins: 113
Ties: 0

Against Seasonal Naive
LoRA wins: 318
Zero-Shot wins: 318


In [21]:
# Calculate percentage change in MAE from zero-shot to LoRA.
# Positive values mean LoRA improved; negative values mean it performed worse.
comparison["LoRA_MAE_Improvement_%"] = (
    (comparison["MAE_chronos"] - comparison["LoRA_MAE"])
    / comparison["MAE_chronos"]
) * 100

print("Mean improvement (%):",
      comparison["LoRA_MAE_Improvement_%"].mean())

print("Median improvement (%):",
      comparison["LoRA_MAE_Improvement_%"].median())

print("\nLargest LoRA improvements:")
display(
    comparison.nlargest(5, "LoRA_MAE_Improvement_%")[
        ["series_id", "MAE_chronos", "LoRA_MAE",
         "LoRA_MAE_Improvement_%"]
    ]
)

print("\nLargest LoRA regressions:")
display(
    comparison.nsmallest(5, "LoRA_MAE_Improvement_%")[
        ["series_id", "MAE_chronos", "LoRA_MAE",
         "LoRA_MAE_Improvement_%"]
    ]
)

Mean improvement (%): -0.0076240348971263324
Median improvement (%): 0.053295397402268886

Largest LoRA improvements:


,series_id,MAE_chronos,LoRA_MAE,LoRA_MAE_Improvement_%
25,T26,33.351749,33.080536,0.813191
81,T82,125.484856,124.687660,0.635292
132,T133,141.100296,140.312378,0.558410
320,T321,127.593994,127.000580,0.465080
128,T129,733.197083,730.093628,0.423277



Largest LoRA regressions:


,series_id,MAE_chronos,LoRA_MAE,LoRA_MAE_Improvement_%
182,T183,9.511116e-10,1.085192e-09,-14.097279
306,T307,1.555690e+02,1.567717e+02,-0.773086
313,T314,1.634958e+04,1.645567e+04,-0.648866
90,T91,1.956723e+01,1.969233e+01,-0.639330
308,T309,5.705316e+02,5.733248e+02,-0.489581


## 5. Saving Final Results

The per-customer comparison, aggregate performance metrics, raw test predictions, and computational cost summaries are saved to Google Drive. This completes the project evaluation deliverables.

In [22]:
results_dir = "/content/drive/MyDrive/Chronos2_Finetune/Final_Results"
os.makedirs(results_dir, exist_ok=True)

# Save per-customer comparison.
comparison.to_csv(
    f"{results_dir}/final_test_comparison.csv",
    index=False
)

# Save aggregate performance metrics.
summary_df.to_csv(
    f"{results_dir}/final_test_summary.csv"
)

# Save LoRA predictions and actual test values.
np.savez_compressed(
    f"{results_dir}/lora_test_predictions.npz",
    predictions=lora_predictions,
    actuals=actuals,
    series_indices=series_indices
)

with open(
    "/content/drive/MyDrive/Chronos2_Finetune/lora_training_results.json",
    "r"
) as f:
    training_results = json.load(f)

# Combine training and test inference costs.
computational_results = {
    "training_time_minutes": training_results["training_time_minutes"],
    "peak_training_gpu_memory_gb": training_results["peak_gpu_memory_gb"],
    "test_inference_time_minutes": inference_time / 60,
    "test_forecast_windows": len(contexts),
    "inference_batch_size": batch_size
}

with open(
    f"{results_dir}/computational_results.json",
    "w"
) as f:
    json.dump(computational_results, f, indent=4)

print("All final results saved.")
print("Location:", results_dir)

All final results saved.
Location: /content/drive/MyDrive/Chronos2_Finetune/Final_Results
